# A two-stage AO system: modulated Pyramid "woofer" + vZWFS "tweeter"

This notebook simulates and trains a **two-stage** AO system: a modulated Pyramid WFS driving a first ("woofer") DM, feeding a vector-Zernike WFS (vZWFS) driving a second ("tweeter") DM. This is a fictional demo instrument built from scratch purely to demonstrate the two-stage architecture -- there is no real bench behind it, so (unlike the other `Tutorials/<Instrument>` notebooks) the WFS and DM below are never loaded from a bench calibration; they simply use their nominal, as-built geometry.

**Architecture:**
- A single shared atmosphere/telescope feeds both stages.
- **Stage 1** (Pyramid + DM1) is a standard closed loop, exactly like the single-stage `Tutorials/<Instrument>` notebooks: WFS1 senses the residual left over from DM1's *own* previous correction, never anything from DM2.
- **Stage 2** (vZWFS + DM2) is fed the phase *after* DM1's correction has been applied, and runs its own independent closed loop on top of that residual.
- Stage 2 runs `N` times faster than stage 1 (`TrainParams['SpeedRatioN']`): DM1's command is only re-measured and updated every `N`-th tick, while DM2 updates every tick.
- Each stage's leaky integrator has its own independently-drawn gain/leak -- they are two physically separate control loops, not a shared one.

**Training strategy:** `AI4AO.Trainer.Trainer` only supports a single WFS/DM pair, so it can't directly chain two stages. Instead we reuse it twice, sequentially:
1. Train stage 1's reconstructor (`CNN1`) completely standalone, exactly like the Rama/Ekarus/Papyrus training notebooks -- a normal single-stage closed loop on the raw atmosphere.
2. Freeze stage 1 (WFS1, DM1, CNN1) and train stage 2's reconstructor (`CNN2`) with a small dataset wrapper (`Stage1ResidualDataset`, defined below) that runs the frozen stage-1 loop internally at the slow rate and hands `Trainer` the post-DM1 residual at the fast rate -- from `Trainer`'s point of view this looks exactly like an ordinary `PhaseDataset`, so no changes to `Trainer` itself are needed.

Each CNN's gradient only ever comes from its own stage's loss -- stage 2's training never backpropagates into stage 1.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import torch
import numpy as np
import torch.nn as nn
import os

from AI4AO import PyramidWFS, ZernikeWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss

device = 'cuda'  # set to "cpu" if CUDA is not available

## Loading the shared configuration

`TwoStageAO_params.py` holds one shared `WFSParams` (telescope + detector-noise + frame-preprocessing settings common to both stages), one shared `AtmosParams`, a `LoopParams` set at the *fast* (stage 2) tick rate, two separate DM dicts (`DMParams1` the woofer, `DMParams2` the tweeter -- different `Nactuator`, so nothing to share there), and `TrainParams` (learning rates, run lengths, the stage-1/stage-2 BPTT window lengths, and `SpeedRatioN`).

We copy `WFSParams` once per stage and add the WFS-specific keys ourselves: `Wavelength`/`Modulation` for the Pyramid, `Wavelength`/`MaskType`/`Use_MTF`/`MTF_upscale` for the vZWFS (`MaskType: "vzwfs"` is the same two-mask code path as Oziriis's `"DoubleZernike"` in `AI4AO/ZernikeWFS.py`).

In [ ]:
paramfile = 'TwoStageAO_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']  # fast (stage 2) tick rate
DMParams1 = Config.fromfile(paramfile)['DMParams1']
DMParams2 = Config.fromfile(paramfile)['DMParams2']
TrainParams = Config.fromfile(paramfile)['TrainParams']

N = TrainParams['SpeedRatioN']  # stage 2 runs N times faster than stage 1

# Stage 1 (modulated Pyramid): its own closed loop runs at loopFrequency / N
WFSParams1 = WFSParams.copy()
WFSParams1.update(Wavelength=650e-9, Modulation=3)

LoopParams1 = LoopParams.copy()
LoopParams1['loopFrequency'] = LoopParams['loopFrequency'] / N

LoopParams2 = LoopParams.copy()
LoopParams2["levelOfCorrection"] = [0.6, 1]

# Stage 2 (vZWFS): senses the residual after DM1, at the fast tick rate
WFSParams2 = WFSParams.copy()
WFSParams2.update(Wavelength=1550e-9, MaskType="vzwfs", Use_MTF=False, MTF_upscale=10)

AtmosParams2 = AtmosParams.copy()
AtmosParams2["r0"] = list(map(lambda x: x * (1550e-9/650e-9)**(6/5), AtmosParams["r0"]))


PATH = "../../Data/TwoStageAO/"
os.makedirs(PATH, exist_ok=True)

## Reconstructor architecture

One `PWFSNet` class, parametrized by input channel count, serves both stages: 4 channels for the Pyramid's 4 pupil images, 2 channels for the vZWFS's 2 pupil images. Each pupil image is processed independently in the stem (`groups=n_channels`) before the encoder mixes information across them, exactly the pattern the other instruments' pyramid networks use -- just generalized to also fit the vZWFS's 2-pupil case.

In [ ]:
class PWFSNet(nn.Module):
    def __init__(self, n_channels, Nmodes):
        super().__init__()

        self.stem = nn.Sequential(
            # Process each pupil image independently
            nn.Conv2d(n_channels, 8 * n_channels, kernel_size=11, padding=5, groups=n_channels),
            nn.GELU(),

            nn.Conv2d(8 * n_channels, 16 * n_channels, kernel_size=7, padding=3, groups=n_channels),
            nn.GELU(),

            nn.MaxPool2d(2),
        )

        self.encoder = nn.Sequential(
            nn.Conv2d(16 * n_channels, 64, kernel_size=5, padding=2),
            nn.GELU(),

            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.GELU(),

            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.GELU(),

            nn.AdaptiveAvgPool2d(1),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, Nmodes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)

## Stage 1: the modulated Pyramid woofer

This is a completely standard single-stage closed loop, built and trained exactly like the `Tutorials/<Instrument>` notebooks -- the only difference is that WFS1/DM1 use their nominal as-built geometry instead of a bench calibration (`LoadCalibration`), since there is no real bench for this fictional instrument. We still freeze both (`.eval()` + `requires_grad_(False)`) before training, since only the reconstructor's weights should be updated here.

`DMParams1['Nmodes']` isn't known ahead of time -- it depends on how many actuator-grid points fall inside `DeformableMirror`'s circular aperture cutoff for `Nactuator=15`, which `DeformableMirror.__init__` computes (and prints) as `dm1.totalAct`. We read it back after construction rather than guessing a number, then use the identity matrix as `M2C1` (a purely zonal actuator basis, exactly as `Tutorials/basics/03_DeformableMirrorAndClosedLoop.ipynb` does) since there's no bench-calibrated KL basis for this instrument.

In [ ]:
dataset1 = PhaseDataset(WFSParams1, AtmosParams, LoopParams1, DMParams1, device)
dataset1.generateClosedLoop = True

wfs1 = PyramidWFS(WFSParams1, device)
wfs1.eval()
wfs1.requires_grad_(False)

dm1 = DeformableMirror(WFSParams1, DMParams1, device)
dm1.eval()
dm1.requires_grad_(False)

DMParams1['Nmodes'] = int(dm1.totalAct.item())
M2C1 = torch.eye(DMParams1['Nmodes'], device=device)

framePreprocessor1 = FramePreprocess(WFSParams1, wfs1, device)
framePreprocessor1.ProcessReference(wfs1.reference_intensity)

In [ ]:
phaseReconstructor1 = PWFSNet(n_channels=4, Nmodes=DMParams1['Nmodes']).to(device=device)

total_params = sum(p.numel() for p in phaseReconstructor1.parameters() if p.requires_grad)
print(f"Stage 1 (Pyramid) reconstructor -- total trainable parameters: {total_params:,}")

## Optimizer, loss, and the Trainer

`LogResidualVarianceLoss` is the same physics-aware loss used throughout this repo: `ln(var(residual phase over the pupil))`, directly tied to residual RMS/Strehl. `AI4AO.Trainer.Trainer` bundles WFS1/DM1/`framePreprocessor1`/`M2C1`/`phaseReconstructor1`/`dataset1` and implements the standard closed-loop training step, exactly as in the single-stage notebooks.

In [ ]:
optimizer1 = torch.optim.AdamW(phaseReconstructor1.parameters(), TrainParams['lrn'], fused=True)
loss1 = LogResidualVarianceLoss(dataset1.pupil)

trainer1 = Trainer(wfs=wfs1,
                    framePreprocessor=framePreprocessor1,
                    dm=dm1,
                    M2C=M2C1,
                    phaseReconstructor=phaseReconstructor1,
                    dataset=dataset1,
                    loss=loss1,
                    optimizer=optimizer1)

try:
    trainer1.load_checkpoint(PATH + "Stage1CNN.pth", load_optimizer=False)
except KeyError:
    print("Starting from scratch")

## Training stage 1

`trainer1.train(training_steps, closed_loop_iterations)` runs `training_steps` closed-loop optimizer updates, backpropagating through `closed_loop_iterations` simulated AO-loop steps per update. Re-run this cell with a different `TrainParams['TrainRunNb1']` to train further without losing the optimizer's momentum state.

In [ ]:
loss_tracker1, loss_tracker1_ideal = trainer1.train(5000, 1)

In [ ]:
trainer1.plot_losses(loss_tracker1, loss_tracker1_ideal)

In [ ]:
trainer1.save_checkpoint(PATH + "Stage1CNN.pth")

## Visualizing stage 1 alone

A quick sanity check before chaining stage 2 on top: `trainer1.evaluate()` runs a no-grad closed-loop rollout (no pupil noise) and returns the phase, residual phase, and WFS frames at every step.

In [ ]:
n_frames = 50
result1 = trainer1.evaluate(n_steps=n_frames, dataset=dataset1)

fig, axes = imshow_multiple(
    [
        {"tensor": result1.phase[0], "title": "Input phase", "same_scale": True},
        {"tensor": result1.residual_phase[0], "title": "Residual after DM1", "scale_reference": result1.phase[0]},
        {"tensor": result1.wfs_frames[0], "title": "WFS1 frame"},
        {"tensor": result1.psfs[0], "title": "PSF", "same_scale": True},
    ],
    max_channel_number=4
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result1.phase[i], "title": "Input phase", "same_scale": True},
            {"tensor": result1.residual_phase[i], "title": "Residual after DM1", "scale_reference": result1.phase[i]},
            {"tensor": result1.wfs_frames[i], "title": "WFS1 frame"},
            {"tensor": result1.psfs[i], "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes,
        max_channel_number=4
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 100
HTML(anim.to_jshtml())

## Stage 2: the vZWFS tweeter, and the dual-rate residual dataset

Stage 2 never sees the raw atmosphere directly -- only what's left after DM1's correction. `Stage1ResidualDataset` below wraps a *raw*-atmosphere `PhaseDataset` (ticking at the fast rate) together with the now-frozen stage-1 loop (`wfs1`/`dm1`/`framePreprocessor1`/`phaseReconstructor1`/`M2C1`), and exposes the exact same `dataset[idx]` contract `PhaseDataset` does -- `Trainer` can consume it without any changes.

On every call it advances the raw atmosphere by one fast tick. Only every `N`-th call (`idx % N == 0`) does it re-run stage 1's own measurement -> leaky-integrator -> DM1 update; in between, DM1's held correction is simply reused against the newly wind-shifted raw phase. The returned `"phase"` is the residual after DM1's correction -- what stage 2 actually has to work with -- and `"loop_gain"`/`"loop_leak"` are stage 2's *own*, independently-drawn integrator gain/leak (not stage 1's), so `Trainer.train()`'s built-in leaky integrator ends up closing stage 2's loop with its own dynamics.

Stage 1's own gain/leak (used internally, for its frozen loop) are redrawn once per rollout from `LoopParams1`'s range -- the same range `CNN1` was trained under in the cell above.

In [ ]:
dataset2 = PhaseDataset(WFSParams1, AtmosParams2, LoopParams2, DMParams1, device)
dataset2.generateClosedLoop = True

wfs2 = ZernikeWFS(WFSParams2, device)
wfs2.eval()

dm2 = DeformableMirror(WFSParams2, DMParams2, device)
dm2.eval()

DMParams2['Nmodes'] = int(dm2.totalAct.item())
M2C2 = torch.eye(DMParams2['Nmodes'], device=device)

framePreprocessor2 = FramePreprocess(WFSParams2, wfs2, device)
framePreprocessor2.ProcessReference(wfs2.reference_intensity)

In [ ]:
phaseReconstructor2 = PWFSNet(n_channels=2, Nmodes=DMParams2['Nmodes']).to(device=device)

total_params = sum(p.numel() for p in phaseReconstructor2.parameters() if p.requires_grad)
print(f"Stage 2 (vZWFS) reconstructor -- total trainable parameters: {total_params:,}")

## Optimizer, loss, and the Trainer for stage 2

Same loss and `Trainer` machinery as stage 1, just pointed at `wfs2`/`dm2`/`framePreprocessor2`/`M2C2`/`phaseReconstructor2`/`dataset2` -- `Trainer` has no idea `dataset2` is secretly running a frozen closed loop internally, it just sees `dataset2[idx]` returning phase/pupil/noise/gain/leak like any other dataset.

In [ ]:
optimizer2 = torch.optim.AdamW(phaseReconstructor2.parameters(), TrainParams['lrn'], fused=True)
loss2 = LogResidualVarianceLoss(dataset1.pupil)

trainer2 = Trainer(wfs=wfs2,
                    framePreprocessor=framePreprocessor2,
                    dm=dm2,
                    M2C=M2C2,
                    phaseReconstructor=phaseReconstructor2,
                    dataset=dataset2,
                    loss=loss2,
                    optimizer=optimizer2)

try:
    trainer2.load_checkpoint(PATH + "Stage2CNN.pth", load_optimizer=False)
except:
    print("Starting from scratch")

## Training stage 2

`TrainParams['ClosedLoopIterations2']` should be at least `N` (and ideally several multiples of it) so each training rollout includes a handful of stage-1 updates, not just a single held DM1 correction.

In [ ]:
loss_tracker2, loss_tracker2_ideal = trainer2.train(5000, 1)

In [ ]:
trainer2.plot_losses(loss_tracker2, loss_tracker2_ideal)

In [ ]:
trainer2.save_checkpoint(PATH + "Stage2CNN.pth")

## Visualizing stage 2 alone

A quick sanity check before chaining the two stages: `trainer2.evaluate()` runs a no-grad closed-loop rollout (no pupil noise) and returns the phase, residual phase, and WFS frames at every step.

In [ ]:
n_frames = 50
result1 = trainer2.evaluate(n_steps=n_frames, dataset=dataset1)

fig, axes = imshow_multiple(
    [
        {"tensor": result1.phase[0], "title": "Input phase", "same_scale": True},
        {"tensor": result1.residual_phase[0], "title": "Residual after DM1", "scale_reference": result1.phase[0]},
        {"tensor": result1.wfs_frames[0], "title": "WFS1 frame"},
        {"tensor": result1.psfs[0], "title": "PSF", "same_scale": True},
    ],
    max_channel_number=4
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result1.phase[i], "title": "Input phase", "same_scale": True},
            {"tensor": result1.residual_phase[i], "title": "Residual after DM1", "scale_reference": result1.phase[i]},
            {"tensor": result1.wfs_frames[i], "title": "WFS1 frame"},
            {"tensor": result1.psfs[i], "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes,
        max_channel_number=4
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 100
HTML(anim.to_jshtml())

## Visualizing the full two-stage closed loop

`trainer2.evaluate(dataset=dataset2)` alone would only show what stage 2 sees (already post-DM1). To see the whole chain -- raw atmosphere, the residual after DM1, and the final residual after DM2 -- we hand-roll a no-grad rollout over a *fresh* raw-atmosphere dataset, running both frozen/trained stages explicitly and recording every intermediate quantity. This mirrors `Stage1ResidualDataset` plus stage 2's own closed loop, but keeps the diagnostics `Trainer.evaluate()` doesn't expose.

In [ ]:
n_frames = 100

viz_dataset = PhaseDataset(WFSParams1, AtmosParams, LoopParams, DMParams1, device)
viz_dataset.Nphases = 4
viz_dataset.generateClosedLoop = False

phaseReconstructor1.eval()
phaseReconstructor2.eval()

raw_phases, post_dm1, post_dm2, wfs1_frames, wfs2_frames, psf1_frames, psf2_frames = [], [], [], [], [], [], []

with torch.no_grad():
    for i in range(n_frames):
        raw_batch = viz_dataset[i]
        raw_phase = raw_batch["phase"]
        pupil = raw_batch["pupil"]

        if i == 0:
            Nphases = raw_phase.shape[0]
            gain1 = torch.empty(Nphases, 1, device=device).uniform_(*LoopParams1['loopGain'])
            leak1 = torch.empty(Nphases, 1, device=device).uniform_(*LoopParams1['loopLeak'])
            gain2 = torch.empty(Nphases, 1, device=device).uniform_(*LoopParams['loopGain'])
            leak2 = torch.empty(Nphases, 1, device=device).uniform_(*LoopParams['loopLeak'])

            z1_estimated = torch.zeros(Nphases, DMParams1['Nmodes'], device=device)
            z1_buffer = torch.zeros_like(z1_estimated)
            z1_output = torch.zeros_like(z1_estimated)
            phase_reconstructed_1 = torch.zeros_like(raw_phase)

            z2_estimated = torch.zeros(Nphases, DMParams2['Nmodes'], device=device)
            z2_buffer = torch.zeros_like(z2_estimated)
            z2_output = torch.zeros_like(z2_estimated)
            phase_reconstructed_2 = torch.zeros_like(raw_phase)

            wfs1.SetPhotonsAndRON(raw_batch["nphotons"], raw_batch["ron"])
            wfs2.SetPhotonsAndRON(raw_batch["nphotons"], raw_batch["ron"])

        # --- Stage 1: only ticks every N frames ---
        if i % 3 == 0:
            residual1 = raw_phase - phase_reconstructed_1
            if i > n_frames * 0.1:
                z1_estimated = z1_estimated * leak1 + gain1 * z1_buffer
            z1_buffer = torch.clone(z1_output)

            wfs1_frame = wfs1(residual1, pupil)
            preprocessed1 = framePreprocessor1.ProcessFrame(wfs1_frame, False)
            z1_output = phaseReconstructor1(preprocessed1)

            phase_reconstructed_1 = dm1(z1_estimated @ M2C1.T)

        psf1_frame = wfs1.GetPSF(residual1, pupil, sampling = 4, fov = 20)
        residual_after_stage1 = raw_phase - phase_reconstructed_1

        # --- Stage 2: ticks every frame ---
        residual2 = residual_after_stage1 - phase_reconstructed_2    #  * (dm1.wavelength / dm2.wavelength)

        if i > n_frames * 0.5:
            z2_estimated = z2_estimated * leak2 + gain2 * z2_buffer
        z2_buffer = torch.clone(z2_output)

        wfs2_frame = wfs2(residual2, pupil)
        preprocessed2 = framePreprocessor2.ProcessFrame(wfs2_frame, False)
        z2_output = phaseReconstructor2(preprocessed2)

        phase_reconstructed_2 = dm2(z2_estimated @ M2C2.T)

        psf2_frame = wfs1.GetPSF(residual2, pupil, sampling = 4, fov = 20)
        residual_after_stage2 = residual_after_stage1 - phase_reconstructed_2

        raw_phases.append(raw_phase)
        post_dm1.append(residual_after_stage1)
        post_dm2.append(residual_after_stage2)
        wfs1_frames.append(wfs1_frame)
        wfs2_frames.append(wfs2_frame)
        psf1_frames.append(psf1_frame)
        psf2_frames.append(psf2_frame)

raw_phases = torch.stack(raw_phases)
post_dm1 = torch.stack(post_dm1)
post_dm2 = torch.stack(post_dm2)
wfs1_frames = torch.stack(wfs1_frames)
wfs2_frames = torch.stack(wfs2_frames)
psf1_frames = torch.stack(psf1_frames)
psf2_frames = torch.stack(psf2_frames)

In [ ]:
fig, axes = imshow_multiple(
    [
        {"tensor": raw_phases[0], "title": "Raw atmosphere"},
        {"tensor": post_dm1[0], "title": "Residual after DM1", "scale_reference": raw_phases[0]},
        {"tensor": post_dm2[0], "title": "Residual after DM2", "scale_reference": raw_phases[0]},
        {"tensor": wfs1_frames[0], "title": "WFS1 (Pyramid) frame"},
        {"tensor": wfs2_frames[0], "title": "WFS2 (vZWFS) frame"},
        {"tensor": psf1_frames[0], "title": "PSF 1"},
        {"tensor": psf2_frames[0], "title": "PSF 2"},
    ],
    max_channel_number=4
)


def update(i):
    imshow_multiple(
        [
            {"tensor": raw_phases[i], "title": "Raw atmosphere"},
            {"tensor": post_dm1[i], "title": "Residual after DM1", "scale_reference": raw_phases[i]},
            {"tensor": post_dm2[i], "title": "Residual after DM2", "scale_reference": raw_phases[i]},
            {"tensor": wfs1_frames[i], "title": "WFS1 (Pyramid) frame"},
            {"tensor": wfs2_frames[i], "title": "WFS2 (vZWFS) frame"},
            {"tensor": psf1_frames[i], "title": "PSF 1"},
            {"tensor": psf2_frames[i], "title": "PSF 2"},
        ],
        fig=fig, axes=axes,
        max_channel_number=4
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 100
HTML(anim.to_jshtml())